![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 03: Big Data)**

**Practice Lab M03B: Player parquet ETL with SQL-style and pandas transformations**

---

- Materials in this module support practical learning in modern data science, big data processing, data acquisition, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 03. The full module is normally completed across two two-hour practical sessions, together with the other notebooks listed in the SIT742 repository.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with Python, pandas, NumPy, pyarrow, and optional pandasql</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>Filtered player data, SQL-style summaries, pandas summaries, one-hot encoded features, and a distance-matrix summary</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development</td>
</tr>
</tbody>
</table>

</div>


<a id="1-overview-and-learning-goals"></a>
### 1. Overview and Learning Goals

This practical uses `player.parquet` to compare SQL-style transformations with pandas transformations. The workflow keeps `pandasql` optional so the notebook can still run in environments where the package is unavailable.

By the end of this lab, students should be able to:

1. load and inspect a parquet dataset;
2. filter records by numeric conditions;
3. compare SQL-style aggregation with pandas `groupby`;
4. create ordered score bands;
5. one-hot encode categorical features; and
6. compute a small Euclidean distance matrix from numeric features.


<a id="2-setup-and-data-files"></a>
### 2. Setup and Data Files

This notebook uses `player.parquet`.

If `pyarrow` is not available, install it in a separate setup cell with `%pip install pyarrow`. If you want to run the optional SQL cells and `pandasql` is unavailable, install it with `%pip install pandasql`; the notebook also provides pandas equivalents.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import json
import sys
import tempfile

import numpy as np
import pandas as pd

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sit742_m03b_output_"))
required_files = ['player.parquet']


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m03b_data_"))
    downloaded_paths = {}
    for filename in required_files:
        local_file = data_dir / filename
        urlretrieve(f"{PUBLIC_DATA_BASE_URL}/{filename}", local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    candidates = [
        Path.cwd() / "Jupyter" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Jupyter" / "data",
    ]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError("Could not find the SIT742 public data folder. Searched:\n" + searched)


if EXECUTION_MODE == "online":
    DATA_DIR, data_paths_by_file = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR = find_local_data_dir(required_files)
    data_paths_by_file = {filename: DATA_DIR / filename for filename in required_files}
else:
    raise ValueError('EXECUTION_MODE must be "online" or "local".')

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)


In [ ]:
try:
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError("This notebook requires pyarrow. In a notebook cell, run `%pip install pyarrow` and then rerun setup.") from exc

try:
    from pandasql import sqldf
except ImportError:
    sqldf = None
    print("pandasql is optional and is not installed; pandas equivalents will still run.")

player_path = data_paths_by_file["player.parquet"]
print("Player parquet:", player_path)


<a id="3-read-and-filter-player-data"></a>
### 3. Read and Filter Player Data


In [ ]:
parquet_file = pq.ParquetFile(player_path)
print(parquet_file.schema)
print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.metadata.num_columns)

player_df = pd.read_parquet(player_path, engine="pyarrow")
player_dict = player_df.to_dict(orient="list")

height_mask = (player_df["Height(CM)"] > 160) & (player_df["Height(CM)"] < 175)
player_filtered = player_df.loc[height_mask].reset_index(drop=True)
player_json_path = OUTPUT_DIR / "newplayer.json"
player_filtered.to_json(player_json_path, orient="records", indent=2)

print("Filtered rows:", len(player_filtered))
player_filtered.head()


<a id="4-sql-style-etl"></a>
### 4. SQL-Style ETL

The optional SQL example renames height and weight columns to SQL-friendly names. If `pandasql` is unavailable, use the pandas equivalent in the next section.


In [ ]:
df_newplayer = pd.read_json(player_json_path)
sql_frame = df_newplayer.rename(columns={"Height(CM)": "Height", "Weight(KG)": "Weight"})

score_columns = [
    "Crossing",
    "Finishing",
    "HeadingAccuracy",
    "ShortPassing",
    "Dribbling",
    "Curve",
    "FKAccuracy",
    "LongPassing",
    "BallControl",
]

if sqldf is not None:
    query1 = sqldf("SELECT * FROM sql_frame LIMIT 10", {"sql_frame": sql_frame})
    query3 = sqldf(
        """
        SELECT Height, Weight,
               AVG(Crossing) AS avg_crossing,
               AVG(Finishing) AS avg_finishing,
               AVG(HeadingAccuracy) AS avg_heading,
               AVG(ShortPassing) AS avg_short_passing,
               AVG(Dribbling) AS avg_dribbling,
               AVG(Curve) AS avg_curve,
               AVG(FKAccuracy) AS avg_fk_accuracy,
               AVG(LongPassing) AS avg_long_passing,
               AVG(BallControl) AS avg_ball_control
        FROM sql_frame
        GROUP BY Height, Weight
        ORDER BY Height, Weight
        """,
        {"sql_frame": sql_frame},
    )
else:
    query1 = sql_frame.head(10)
    query3 = None

query1


<a id="5-pandas-equivalent"></a>
### 5. pandas Equivalent

Now repeat the transformation with pandas. This is the routine path that should work in any environment with pandas and parquet support.


In [ ]:
def label_score(series):
    return pd.cut(
        series,
        bins=[0, 27, 61, 73, 100],
        labels=["low", "medium", "good", "excellent"],
        include_lowest=True,
    )


df_sel = df_newplayer[["Height(CM)", "Weight(KG)", *score_columns]].copy()
df_grouped = (
    df_sel.groupby(["Height(CM)", "Weight(KG)"], as_index=False)[score_columns]
    .mean()
    .sort_values(["Height(CM)", "Weight(KG)"])
)
df_grouped["Dribbling band"] = label_score(df_grouped["Dribbling"])
df_grouped["BallControl band"] = label_score(df_grouped["BallControl"])

print("Grouped rows:", len(df_grouped))
df_grouped.head()


<a id="6-one-hot-and-distance"></a>
### 6. One-Hot Encoding and Distance Matrix


In [ ]:
num_col = df_grouped.select_dtypes(include=np.number).columns
cat_col = df_grouped.select_dtypes(exclude=np.number).columns

df_num = df_grouped[num_col]
df_cat = df_grouped[cat_col]
df_onehot = pd.get_dummies(df_cat, dtype=int)
df_all = pd.concat([df_num.reset_index(drop=True), df_onehot.reset_index(drop=True)], axis=1)

print("Prepared feature shape:", df_all.shape)
df_all.head()


In [ ]:
def distance_to_one(row1, row2):
    return np.sqrt(np.sum(np.square(row1 - row2)))


def distance_to_many(row1, rows):
    return np.sqrt(np.sum(np.square(rows - row1), axis=1))


def nearest_neighbour_summary(frame, sample_size=100):
    sample = frame.head(sample_size).to_numpy(dtype=float)
    distance_matrix = np.vstack([distance_to_many(row, sample) for row in sample])
    nearest = []
    for index, distances in enumerate(distance_matrix):
        order = np.argsort(distances)
        nearest.append(order[1] if len(order) > 1 else order[0])
    return distance_matrix, pd.DataFrame({"index": range(len(nearest)), "most_similar_index": nearest})


In [ ]:
distance_matrix, nearest = nearest_neighbour_summary(df_all, sample_size=min(100, len(df_all)))
print("Distance matrix shape:", distance_matrix.shape)
nearest.head(10)


<a id="7-student-tasks"></a>
### 7. Student Tasks

1. Change the height filter to a different 10 cm interval and report the row count.
2. Add `Acceleration` and `SprintSpeed` to the aggregation list.
3. Compare one row from the SQL-style output with the pandas output.
4. Explain why score bands are categorical rather than numeric.
5. Identify the row with the highest average `BallControl`.


In [ ]:
# Task workspace.
height_175_185 = player_df.loc[(player_df["Height(CM)"] > 175) & (player_df["Height(CM)"] < 185)]
best_ball_control = df_grouped.sort_values("BallControl", ascending=False).head(1)

print("Rows for height 175 to 185:", len(height_175_185))
best_ball_control[["Height(CM)", "Weight(KG)", "BallControl", "BallControl band"]]


<a id="8-checks"></a>
### 8. Checks


In [ ]:
assert player_df.shape[0] == 18159
assert player_json_path.exists()
assert len(player_filtered) > 0
assert df_grouped["Dribbling band"].notna().all()
assert df_all.shape[0] == df_grouped.shape[0]
assert distance_matrix.shape[0] == distance_matrix.shape[1]

print("M03PracClass-B checks passed.")


<a id="9-reflection-and-references"></a>
### 9. Reflection and References

Reflection prompts:

1. Which transformation was clearer in SQL-style syntax?
2. Which transformation was clearer in pandas?
3. Why is package availability part of environment-aware planning?

References:

- pandas documentation: `read_parquet`, `groupby`, `cut`, `get_dummies`
- pandasql project documentation
- Apache Parquet documentation
